In [ ]:
%%bash
set -e
CODE=$(dirname "$(find /kaggle/input -name dividemix.py | head -1)")
DATA=$(dirname "$(find /kaggle/input -name train_images.npy | head -1)")
echo "code=$CODE data=$DATA"; nvidia-smi -L
rm -rf /kaggle/working/cv && cp -r "$CODE" /kaggle/working/cv
ln -sfn "$DATA" /kaggle/working/data

In [ ]:
%%bash
set -e
# pipeline smoke test (4 tiny epochs incl. 2 warm-up, 4096 images, bs 128): fails fast on bugs; outputs discarded
cd /kaggle/working/cv
python dividemix.py --data ../data --out /tmp/smoke --label_col noisy_label --smoke
python make_preds.py --run /tmp/smoke --epoch best --out /tmp/smoke/p.csv
python -c "import pandas as pd; d=pd.read_csv('/tmp/smoke/log_full.csv'); print(d.to_string()); r=d[d.phase=='dividemix']; assert len(r)==2 and (r.net0_fallback==0).all() and (r.net1_fallback==0).all(), 'DivideMix path not exercised'; print('SMOKE OK')"
rm -rf /tmp/smoke

In [ ]:
%%bash
# T4 DivideMix on noisy labels (GPU0) and the same method on synthetic labels for T5 (GPU1)
cd /kaggle/working/cv
CUDA_VISIBLE_DEVICES=0 python dividemix.py --data ../data --out ../outputs/t4_dividemix       --label_col noisy_label     > ../t4_dividemix.log 2>&1 &
CUDA_VISIBLE_DEVICES=1 python dividemix.py --data ../data --out ../outputs/t5_dividemix_synth --label_col synthetic_label > ../t5_dividemix_synth.log 2>&1 &
wait
tail -n 2 ../t4_dividemix.log ../t5_dividemix_synth.log

In [ ]:
%%bash
cd /kaggle/working/cv
for r in t4_dividemix t5_dividemix_synth; do
  python make_preds.py --run ../outputs/$r --epoch final --out ../predictions/${r}_final.csv
  python make_preds.py --run ../outputs/$r --epoch best  --out ../predictions/${r}_best.csv
done
rm -f /kaggle/working/data